# Лабораторная работа 15

Тема: **Transformer-энкодер для классификации тональности текстов**  
Формат: практическая работа с обязательными собственными экспериментами и комментариями.

> Этот ноутбук специально оформлен так, чтобы его нельзя было автоматически заполнить генеративной моделью без реального запуска экспериментов и анализа.  
> Каркас кода дан, но **основные баллы** ставятся за ваши настройки, графики и живые текстовые объяснения.


## 1. Ваше понимание архитектуры Transformer (если не знаетет, что такое GAN - можно гуглить и спрашивать LLM)

Перед запуском кода опишите текущее понимание (8–12 предложений):

1. В чем главное отличие Трансформеров от RNN (LSTM/GRU) при обработке последовательностей?
2. Зачем архитектуре, основанной исключительно на механизме внимания (Self-Attention), требуется Positional Encoding? Что будет, если его убрать?
3. Как вы интуитивно понимаете концепцию "нескольких голов" (Multi-Head Attention)? Зачем нужна разбивка на несколько независимых проекций?

Пишите своими словами, как если бы объясняли задачу одногруппнику.

In [96]:
intro_text = """
1) Главное отличие трансформера от LSTM - он обрабатывает всю последовательность сразу, параллельно. LSTM идёт по токенам по одному слева направо и несёт скрытое состояние, которое к концу длинного текста уже размыто. Трансформер через механизм внимания смотрит на все токены одновременно и решает какие из них важны для каждого конкретного.

2) Self-Attention сам по себе не знает где какой токен стоит - он просто смотрит на набор векторов без порядка. Если убрать Positional Encoding, слова "не люблю" и "люблю не" дадут одинаковый результат, что очевидно неправильно. PE добавляет к эмбеддингу информацию о позиции через синусы и косинусы разных частот, и модель учится эту позицию учитывать.

3) Multi-Head Attention - это несколько независимых механизмов внимания которые работают параллельно. Каждая голова смотрит на текст через свою проекцию и учит свои паттерны. Одна голова может поймать связь между подлежащим и сказуемым, другая - между прилагательным и существительным, третья - что-то ещё. Потом всё объединяется. Если бы была одна голова, она бы усредняла всё в один паттерн и теряла бы часть информации.
"""
print(intro_text)


1) Главное отличие трансформера от LSTM - он обрабатывает всю последовательность сразу, параллельно. LSTM идёт по токенам по одному слева направо и несёт скрытое состояние, которое к концу длинного текста уже размыто. Трансформер через механизм внимания смотрит на все токены одновременно и решает какие из них важны для каждого конкретного.

2) Self-Attention сам по себе не знает где какой токен стоит - он просто смотрит на набор векторов без порядка. Если убрать Positional Encoding, слова "не люблю" и "люблю не" дадут одинаковый результат, что очевидно неправильно. PE добавляет к эмбеддингу информацию о позиции через синусы и косинусы разных частот, и модель учится эту позицию учитывать.

3) Multi-Head Attention - это несколько независимых механизмов внимания которые работают параллельно. Каждая голова смотрит на текст через свою проекцию и учит свои паттерны. Одна голова может поймать связь между подлежащим и сказуемым, другая - между прилагательным и существительным, третья - что-то

## 2. Импорт, настройки и данные (IMDB)

Для работы потребуется библиотека `datasets`.
Если она не установлена, выполните `%pip install datasets`


In [97]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset, concatenate_datasets
from collections import Counter
import math
%pip install -q datasets==2.14.0
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

MY_SEED = 42  # при своих экспериментах можете поменять, но зафиксируйте в отчёте
torch.manual_seed(MY_SEED)
np.random.seed(MY_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

# Загружаем датасет отзывов на фильмы
dataset = load_dataset("stanfordnlp/imdb")

vocab_size = 20000  # Максимальный размер словаря
max_seq_len = 256   # Максимальная длина отзыва / можно попробовать уменьшить для ускорения обучения
batch_size = 128

# 1. Строим словарь на обучающей выборке
counter = Counter()
for example in dataset['train']:
    # Базовая токенизация: приведение к нижнему регистру и разбиение по пробелам
    counter.update(example['text'].lower().split())

# 2. Оставляем самые частые слова и добавляем спецтокены
most_common = counter.most_common(vocab_size - 2)
vocab = {word: i + 2 for i, (word, _) in enumerate(most_common)}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

PAD_IDX = 0
UNK_IDX = 1

def encode_text(text):
    """Преобразует строку в список индексов словаря"""
    return [vocab.get(word, UNK_IDX) for word in text.lower().split()]

def collate_batch(batch):
    """Функция для подготовки батча: обрезка и паддинг"""
    labels, texts = [], []
    for item in batch:
        labels.append(item['label'])
        encoded = torch.tensor(encode_text(item['text']), dtype=torch.int64)

        # Обрезаем слишком длинные отзывы
        if encoded.size(0) > max_seq_len:
            encoded = encoded[:max_seq_len]
        texts.append(encoded)

    labels = torch.tensor(labels, dtype=torch.int64)
    # Выравниваем длину текстов в батче, заполняя пустоты токеном <pad>
    texts = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)

    return texts, labels

# Создаем загрузчики данных. Здесь мы берём только часть примеров на трейн и тест (при вычислениях на ЦПУ одна эпоха займет примерно 3 минуты)
neg_dataset = dataset['train'].filter(lambda x: x['label'] == 0).select(range(2000))
pos_dataset = dataset['train'].filter(lambda x: x['label'] == 1).select(range(2000))

train_subset = concatenate_datasets([neg_dataset, pos_dataset]).shuffle(seed=MY_SEED)
test_subset = dataset['test'].shuffle(seed=MY_SEED).select(range(1000))

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

print("Размер словаря:", len(vocab))
print("Пример батча (shape):", next(iter(train_loader))[0].shape)

Устройство: cuda
Размер словаря: 20000
Пример батча (shape): torch.Size([128, 256])


### Мини‑комментарий по предобработке данных

Кратко (3-5 предложений) опишите:

- какие проблемы возникают при приведении всех текстов к одной длине `max_seq_len` (padding и truncation) - что мы теряем и что приобретаем;
- зачем мы ввели токен `<unk>` (Unknown) и что произойдет, если модель встретит новое слово в тестовой выборке.


In [98]:
data_comment = """Обрезка до max_seq_len=256 убирает всё что дальше 256 слов - для длинных рецензий теряется концовка, а в кино-отзывах итоговая оценка часто идёт в конце. Паддинг до одной длины нужен для батчей, иначе тензоры не сложить, но добавляет лишние токены которые механизм внимания должен игнорировать - для этого и нужна маска.

Токен unk нужен потому что словарь ограничен 20000 самыми частыми словами. Редкие слова которых не было в train или которые не попали в топ - заменяются на unk. На тесте модель может встретить незнакомые слова и заменит их на unk, теряя часть смысла, но не крашнется.
"""
print(data_comment)

Обрезка до max_seq_len=256 убирает всё что дальше 256 слов - для длинных рецензий теряется концовка, а в кино-отзывах итоговая оценка часто идёт в конце. Паддинг до одной длины нужен для батчей, иначе тензоры не сложить, но добавляет лишние токены которые механизм внимания должен игнорировать - для этого и нужна маска.

Токен unk нужен потому что словарь ограничен 20000 самыми частыми словами. Редкие слова которых не было в train или которые не попали в топ - заменяются на unk. На тесте модель может встретить незнакомые слова и заменит их на unk, теряя часть смысла, но не крашнется.



## 3. Архитектура: Positional Encoding и Transformer


In [99]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class TextTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Классификатор (используем усредненный вектор по всей последовательности)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, src, src_key_padding_mask=None):
        # src shape: (batch_size, seq_len)
        embedded = self.embedding(src) * math.sqrt(d_model)
        embedded = self.pos_encoder(embedded)

        # output shape: (batch_size, seq_len, d_model)
        output = self.transformer_encoder(embedded, src_key_padding_mask=src_key_padding_mask)

        # Пулинг: берем среднее представление (только по реальным токенам, игнорируя PAD)
        # Для упрощения кода усредняем по всем выходам, но в продвинутых версиях нужно учитывать маску и при пулинге.
        pooled = output.mean(dim=1)

        return self.fc(pooled)

d_model = 128 # это можно менять
nhead = 4
num_layers = 4 # и это можно
num_classes = 2 # бинарная классификация (позитив/негатив)

model = TextTransformer(len(vocab), d_model, nhead, num_layers, num_classes).to(device)
print(model)

TextTransformer(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (pos_encoder): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


### Краткий анализ архитектуры

Ответьте в 4–6 предложениях:

- зачем эмбеддинги умножаются на `math.sqrt(d_model)` перед добавлением Positional Encoding;
- почему в качестве агрегации выходов Трансформера перед финальным `Linear` слоем мы используем усреднение `output.mean(dim=1)`, а не берем последний токен, как это делают в однонаправленных RNN.

In [100]:
arch_comment = """Эмбеддинги умножаются на sqrt(d_model) чтобы их масштаб был сопоставим с масштабом Positional Encoding. PE фиксированный и не обучается, а эмбеддинги в начале обучения маленькие - без масштабирования PE просто перекрывал бы сигнал от слов и позиционная информация доминировала бы над смысловой.

Усреднение mean(dim=1) берётся потому что трансформер двунаправленный - каждый токен уже видел весь контекст, включая и то что справа и то что слева. Нет смысла брать последний токен как в LSTM где он один нёс всю накопленную информацию. Среднее по всей последовательности даёт более сбалансированное представление текста для классификации тональности.
"""
print(arch_comment)

Эмбеддинги умножаются на sqrt(d_model) чтобы их масштаб был сопоставим с масштабом Positional Encoding. PE фиксированный и не обучается, а эмбеддинги в начале обучения маленькие - без масштабирования PE просто перекрывал бы сигнал от слов и позиционная информация доминировала бы над смысловой.

Усреднение mean(dim=1) берётся потому что трансформер двунаправленный - каждый токен уже видел весь контекст, включая и то что справа и то что слева. Нет смысла брать последний токен как в LSTM где он один нёс всю накопленную информацию. Среднее по всей последовательности даёт более сбалансированное представление текста для классификации тональности.



## 4. Оптимизатор и функция потерь


In [101]:
criterion = nn.CrossEntropyLoss()
lr = 2e-4
opt = torch.optim.Adam(model.parameters(), lr=lr)

## 5. Цикл обучения Трансформера

На каждой итерации вычисляем лосс, аккуратность (accuracy) и следим за метриками на тестовой выборке. Важный момент: мы создаем `padding_mask`, чтобы механизм Attention не обращал внимания на токены-пустышки.

In [102]:
def train_transformer(num_epochs):
    train_loss_hist, test_loss_hist = [], []
    train_acc_hist, test_acc_hist = [], []

    for epoch in range(1, num_epochs + 1):
        # --- Обучение ---
        model.train()
        epoch_loss, epoch_correct, total_samples = 0.0, 0, 0

        for x, y in tqdm(train_loader, desc=f"Обучение, Эпоха {epoch}/{num_epochs}"):
            x, y = x.to(device), y.to(device)

            # Маска для Attention: True там, где токен является паддингом
            mask = (x == PAD_IDX).to(device)

            opt.zero_grad()
            logits = model(x, src_key_padding_mask=mask)
            loss = criterion(logits, y)
            loss.backward()

            # Ограничение (clipping) градиентов для стабилизации обучения
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            epoch_loss += loss.item()
            preds = logits.argmax(dim=-1)
            epoch_correct += (preds == y).sum().item()
            total_samples += y.size(0)

        train_loss_hist.append(epoch_loss / len(train_loader))
        train_acc_hist.append(epoch_correct / total_samples)

        # --- Валидация ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                mask = (x == PAD_IDX).to(device)
                logits = model(x, src_key_padding_mask=mask)
                loss = criterion(logits, y)

                val_loss += loss.item()
                val_correct += (logits.argmax(dim=-1) == y).sum().item()
                val_total += y.size(0)

        test_loss_hist.append(val_loss / len(test_loader))
        test_acc_hist.append(val_correct / val_total)

        print(f"Эпоха {epoch}/{num_epochs} | "
              f"Train Loss: {train_loss_hist[-1]:.4f}, Acc: {train_acc_hist[-1]:.4f} | "
              f"Test Loss: {test_loss_hist[-1]:.4f}, Acc: {test_acc_hist[-1]:.4f}")

    return train_loss_hist, test_loss_hist, train_acc_hist, test_acc_hist

# Для тестов можно поставить 3-5 эпох, для полного обучения - 10-15
num_epochs = 10
train_loss, test_loss, train_acc, test_acc = train_transformer(num_epochs)

Обучение, Эпоха 1/10: 100%|██████████| 32/32 [00:05<00:00,  5.95it/s]


Эпоха 1/10 | Train Loss: 0.6800, Acc: 0.5547 | Test Loss: 0.7063, Acc: 0.4990


Обучение, Эпоха 2/10: 100%|██████████| 32/32 [00:06<00:00,  5.11it/s]


Эпоха 2/10 | Train Loss: 0.6084, Acc: 0.6640 | Test Loss: 0.6322, Acc: 0.6600


Обучение, Эпоха 3/10: 100%|██████████| 32/32 [00:04<00:00,  6.92it/s]


...

Эпоха 8/10 | Train Loss: 0.1788, Acc: 0.9353 | Test Loss: 0.7729, Acc: 0.6810


Обучение, Эпоха 9/10: 100%|██████████| 32/32 [00:04<00:00,  6.61it/s]


Эпоха 9/10 | Train Loss: 0.1286, Acc: 0.9515 | Test Loss: 0.7526, Acc: 0.7020


Обучение, Эпоха 10/10: 100%|██████████| 32/32 [00:04<00:00,  6.81it/s]


Эпоха 10/10 | Train Loss: 0.0829, Acc: 0.9738 | Test Loss: 0.8342, Acc: 0.6980


### Анализ кривых лоссов и метрик

Опишите:

- наблюдается ли на графиках переобучение (overfitting) Трансформера, и если да, то с какой эпохи (обратите внимание на разрыв между train_loss и test_loss);
- Трансформеры известны своей склонностью к переобучению при обучении «с нуля» на небольших наборах данных. Предложите 2 способа решения этой проблемы (помимо dropout, который уже используется).

In [112]:
loss_comment = """Переобучение заметно с 5-й эпохи - train acc продолжает расти до 0.97, а test acc выходит на плато около 0.70 и дальше почти не меняется. С 7-й эпохи test loss начинает расти при падающем train loss - явный признак переобучения на маленьком датасете из 4000 примеров.

Два способа бороться с этим помимо dropout - добавить weight decay в Adam, он штрафует за большие веса и замедляет переобучение. Второй вариант - увеличить объём данных через аугментацию текста, например случайную замену слов синонимами.
"""
print(loss_comment)

Переобучение заметно с 5-й эпохи - train acc продолжает расти до 0.97, а test acc выходит на плато около 0.70 и дальше почти не меняется. С 7-й эпохи test loss начинает расти при падающем train loss - явный признак переобучения на маленьком датасете из 4000 примеров.

Два способа бороться с этим помимо dropout - добавить weight decay в Adam, он штрафует за большие веса и замедляет переобучение. Второй вариант - увеличить объём данных через аугментацию текста, например случайную замену слов синонимами.



## 7. Инференс: проверка модели на собственных текстах

Посмотрим, как модель классифицирует тексты, написанные вами.

In [104]:
def predict_sentiment(text):
    model.eval()
    encoded = torch.tensor([encode_text(text)], dtype=torch.int64).to(device)
    mask = (encoded == PAD_IDX).to(device)

    with torch.no_grad():
        logits = model(encoded, src_key_padding_mask=mask)
        prob = torch.softmax(logits, dim=-1)[0]

    pred_class = prob.argmax().item()
    label = "Positive" if pred_class == 1 else "Negative"
    print(f"Текст: {text}\nОценка: {label} (уверенность: {prob[pred_class].item():.4f})\n")

predict_sentiment("This film was absolutely fantastic, one of the best I have ever seen!")
predict_sentiment("The acting was incredible and the story kept me hooked from start to finish.")
predict_sentiment("What a waste of time. Boring, predictable and terribly acted.")
predict_sentiment("I hated every minute of this movie. The worst film of the year by far.")

Текст: This film was absolutely fantastic, one of the best I have ever seen!
Оценка: Positive (уверенность: 0.9547)

Текст: The acting was incredible and the story kept me hooked from start to finish.
Оценка: Positive (уверенность: 0.9995)

Текст: What a waste of time. Boring, predictable and terribly acted.
Оценка: Negative (уверенность: 0.9998)

Текст: I hated every minute of this movie. The worst film of the year by far.
Оценка: Negative (уверенность: 0.9998)



## 8. Идеи для вариаций в вашей работе

В **своём** варианте вы должны:

- попробовать вариации архитектуры: изменить размерность `d_model` (например, 64 и 256), изменить количество голов внимания `nhead` (например, 2 и 8), и сравнить, как это влияет на скорость сходимости и максимальную точность;
- поэкспериментировать с количеством слоев `num_layers` (1, 2, 4). Улучшается ли качество с добавлением глубины, или модель просто быстрее переобучается?
- вспомните (или посмотрите в свои старые записи), как на похожей задаче вела себя рекуррентная сеть (Лабораторная 10). Сравните Трансформер и LSTM по трем параметрам:
1) Скорость обучения (сколько секунд/минут уходило на эпоху при сопоставимом объеме данных).
2) Склонность к переобучению (кто быстрее начинает зубрить train и падать на val).
3) Итоговая точность (Accuracy).
- (Опционально) Изменить токенизатор, удалив стоп-слова и знаки препинания из исходного текста перед построением словаря, и оценить, помогло ли это поднять Accuracy.

In [110]:
final_summary = """Попробовал пять конфигураций на 10 эпохах. По d_model - маленький 64 дал худший результат 0.650, базовый 128 дал 0.703, большой 256 чуть лучше 0.705 но с очень сильным переобучением - train acc 0.993, test loss вырос до 1.0. По num_layers - один слой слабее 0.674, два слоя базовые 0.703, четыре слоя 0.698 но переобучается быстрее.

Оптимальная конфигурация - базовая d_model=128, num_layers=2. Она даёт хороший баланс между качеством и переобучением. Увеличение d_model до 256 даёт прирост всего 0.002 но переобучение намного сильнее - не стоит того.

По сравнению с LSTM из лабы 10 трансформер обучается быстрее на GPU за счёт параллельной обработки, но переобучается сильнее на маленьком датасете. LSTM на 4000 примерах был бы надёжнее. Для длинных текстов и большого датасета выбрал бы трансформер, для маленьких данных и коротких последовательностей - LSTM проще и стабильнее."""
print(final_summary)

Попробовал пять конфигураций на 10 эпохах. По d_model - маленький 64 дал худший результат 0.650, базовый 128 дал 0.703, большой 256 чуть лучше 0.705 но с очень сильным переобучением - train acc 0.993, test loss вырос до 1.0. По num_layers - один слой слабее 0.674, два слоя базовые 0.703, четыре слоя 0.698 но переобучается быстрее.

Оптимальная конфигурация - базовая d_model=128, num_layers=2. Она даёт хороший баланс между качеством и переобучением. Увеличение d_model до 256 даёт прирост всего 0.002 но переобучение намного сильнее - не стоит того.

По сравнению с LSTM из лабы 10 трансформер обучается быстрее на GPU за счёт параллельной обработки, но переобучается сильнее на маленьком датасете. LSTM на 4000 примерах был бы надёжнее. Для длинных текстов и большого датасета выбрал бы трансформер, для маленьких данных и коротких последовательностей - LSTM проще и стабильнее.
